In [4]:
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

class ARDCGovernor:
    def __init__(self, model_id="Qwen/Qwen1.5-0.5B-Chat", device="cuda"):
        self.device = device if torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, device_map=self.device,
            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32
        )
        self.history_C = []
        self.history_mu = []

    def calculate_uncertainty(self, logits):
        probs = F.softmax(logits[:, -1, :], dim=-1)
        return -torch.sum(probs * torch.log(probs + 1e-10), dim=-1).item()

    def calculate_clarity(self, logits):
        probs = F.softmax(logits[:, -1, :], dim=-1)
        return torch.max(probs, dim=-1).values.item()

    def check_stop_law(self, C_n, mu_n, n, threshold=0.05, grace_period=15):
        self.history_C.append(C_n)
        self.history_mu.append(mu_n)

        # FIX 1: The Grace Period. Let the model breathe before auditing.
        if n <= grace_period:
            return False, 0.0

        delta_C = self.history_C[-1] - self.history_C[-2]
        delta_mu = self.history_mu[-1] - self.history_mu[-2]

        safe_delta_mu = delta_mu if delta_mu != 0 else 1e-5
        marginal_efficiency = delta_C / abs(safe_delta_mu)

        if delta_mu > 0 and marginal_efficiency < threshold:
            return True, marginal_efficiency
        return False, marginal_efficiency

    def generate(self, prompt, max_tokens=150, stop_threshold=0.01, grace_period=15):
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        input_ids = inputs.input_ids

        generated_tokens = []
        past_key_values = None

        for n in range(1, max_tokens + 1):
            with torch.no_grad():
                outputs = self.model(input_ids=input_ids, past_key_values=past_key_values, use_cache=True)

            logits = outputs.logits
            past_key_values = outputs.past_key_values

            mu_n = self.calculate_uncertainty(logits)
            C_n = self.calculate_clarity(logits)

            stop_triggered, efficiency = self.check_stop_law(C_n, mu_n, n, stop_threshold, grace_period)

            next_token_id = torch.argmax(logits[:, -1, :], dim=-1).unsqueeze(-1)
            generated_tokens.append(next_token_id.item())
            input_ids = next_token_id

            if stop_triggered or next_token_id.item() == self.tokenizer.eos_token_id:
                break

        self.history_C.clear()
        self.history_mu.clear()
        return self.tokenizer.decode(generated_tokens, skip_special_tokens=True)

# --- BENCHMARK SCRIPT ---
print("Loading MATH-500 dataset...")
dataset = load_dataset("HuggingFaceH4/MATH-500", split="test").select(range(10))

# FIX 2: True Control (ARDC fully disabled) vs Governed
test_configs = {
    "Control (ARDC OFF)": -999.0,
    "Governed (ARDC ON)": -0.15   # Slightly looser threshold to allow complex math steps
}

results = {config: {"total_tokens": 0, "correct": 0} for config in test_configs}

def check_answer(model_output, correct_answer):
    return str(correct_answer).lower() in model_output.lower()

governor = ARDCGovernor(model_id="Qwen/Qwen1.5-0.5B-Chat")
print("\n--- BEGINNING BATCH TEST ---\n")

for config_name, threshold in test_configs.items():
    print(f"\n>>> Running Config: {config_name}")

    for i, item in enumerate(dataset):
        # FIX 3: Prompt Engineering - Force the model to state its final answer clearly
        raw_prompt = f"Solve this math problem step-by-step. Put your final answer inside a \\boxed{{}}.\nProblem: {item['problem']}\nAnswer:"

        messages = [{"role": "user", "content": raw_prompt}]
        prompt = governor.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        # Give it a 15 token grace period to start the thought
        output = governor.generate(prompt, max_tokens=150, stop_threshold=threshold, grace_period=15)

        tokens_used = len(governor.tokenizer.encode(output))
        results[config_name]["total_tokens"] += tokens_used

        is_correct = check_answer(output, item['answer'])
        if is_correct:
            results[config_name]["correct"] += 1

        print(f"[{i+1}/10] {'CORRECT  ' if is_correct else 'INCORRECT'} | Tokens: {tokens_used}")

print("\n" + "="*50)
print("🏆 FINAL BENCHMARK METRICS 🏆")
print("="*50)
for config_name, data in results.items():
    print(f"\n{config_name}:\n  Total Tokens: {data['total_tokens']}\n  Accuracy:     {data['correct']} / 10")

control_tokens = results["Control (ARDC OFF)"]["total_tokens"]
governed_tokens = results["Governed (ARDC ON)"]["total_tokens"]

if control_tokens > 0:
    savings = ((control_tokens - governed_tokens) / control_tokens) * 100
    print(f"\n📉 ARDC Compute Reduction: {savings:.2f}% fewer tokens used.")

Loading MATH-500 dataset...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


--- BEGINNING BATCH TEST ---


>>> Running Config: Control (ARDC OFF)
[1/10] INCORRECT | Tokens: 150
[2/10] INCORRECT | Tokens: 150
[3/10] INCORRECT | Tokens: 150
[4/10] CORRECT   | Tokens: 125
[5/10] INCORRECT | Tokens: 150
[6/10] INCORRECT | Tokens: 150
[7/10] INCORRECT | Tokens: 129
[8/10] INCORRECT | Tokens: 150
[9/10] INCORRECT | Tokens: 150
[10/10] CORRECT   | Tokens: 150

>>> Running Config: Governed (ARDC ON)
[1/10] INCORRECT | Tokens: 17
[2/10] INCORRECT | Tokens: 16
[3/10] INCORRECT | Tokens: 16
[4/10] CORRECT   | Tokens: 18
[5/10] INCORRECT | Tokens: 16
[6/10] INCORRECT | Tokens: 19
[7/10] INCORRECT | Tokens: 18
[8/10] INCORRECT | Tokens: 17
[9/10] INCORRECT | Tokens: 22
[10/10] CORRECT   | Tokens: 19

🏆 FINAL BENCHMARK METRICS 🏆

Control (ARDC OFF):
  Total Tokens: 1454
  Accuracy:     2 / 10

Governed (ARDC ON):
  Total Tokens: 178
  Accuracy:     2 / 10

📉 ARDC Compute Reduction: 87.76% fewer tokens used.
